In [1]:
# Pandas & numpy umumnya sudah tersedia default di Google Colab.
# Cell ini memastikan versi terpasang tanpa mengubah environment lain.
!pip install -q pandas numpy

In [2]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 2.2.2
numpy : 2.0.2


In [3]:
# =====================================================================
# KONFIGURASI — mapping kolom SUDAH DIKONFIRMASI, tidak perlu diubah
# kecuali nama file berbeda dari yang diunggah.
# =====================================================================
CONFIG = {
    "ogimet": {
        "path": "ogimet_dataset.csv",
        "delimiter": ";",
        "encoding": "latin-1",
        "date_col": "Date",
        # nama_variabel_playbook -> nama_kolom_aktual
        "column_map": {
            "RR": "Daily Rainfall (mm)",
            "Tavg": "Average Air Temperature (°C)",
            "RH": "Relative Humidity (%)",
        },
        "missing_placeholders": ["----", "-", "NA", "N/A", "null", ""],
    },
    "sounderpy": {
        "path": "indeks_atmosfer_dataset.csv",
        "delimiter": ";",
        "encoding": "latin-1",
        "date_col": "nominal_date",
        "hour_col": "observation_hour",
        "status_col": "status",
        "column_map": {
            "CIN": "SBCIN",
            "K-Index": "KINDEX",
            "LI": "LI_SB_500",
            "TT": "TT",
            "SWEAT": "SWEAT",
        },
        "missing_placeholders": ["----", "-", "NA", "N/A", "null", ""],
    },
    "period": {"start_year": 2017, "end_year": 2024},
    "missing_significant_threshold": 0.30,
    "output_path": "01_audit_report.csv",
}

print("Konfigurasi dimuat. Mapping kolom mengacu pada hasil validasi struktur yang sudah dikonfirmasi.")

Konfigurasi dimuat. Mapping kolom mengacu pada hasil validasi struktur yang sudah dikonfirmasi.


In [4]:
# CATATAN PENTING: TAHAP 1 BERSIFAT READ-ONLY.
# df_ogimet & df_sounderpy hanya dibaca dan diperiksa di seluruh notebook ini,
# TIDAK PERNAH ditimpa balik ke file asal, tidak diubah nilainya, tidak dihapus barisnya.

load_status = {}

def load_raw_csv(cfg, source_name):
    path = cfg["path"]
    if not os.path.exists(path):
        print(f"[GAGAL] {source_name}: file '{path}' tidak ditemukan.")
        load_status[source_name] = False
        return None
    try:
        df = pd.read_csv(path, delimiter=cfg["delimiter"], encoding=cfg["encoding"])
        print(f"[OK] {source_name}: dimuat dari '{path}' -> {df.shape[0]} baris, {df.shape[1]} kolom")
        load_status[source_name] = True
        return df
    except Exception as e:
        print(f"[GAGAL] {source_name}: error saat membaca '{path}' -> {e}")
        load_status[source_name] = False
        return None


df_ogimet = load_raw_csv(CONFIG["ogimet"], "Ogimet")
df_sounderpy = load_raw_csv(CONFIG["sounderpy"], "SounderPy")

[OK] Ogimet: dimuat dari 'ogimet_dataset.csv' -> 3287 baris, 7 kolom
[OK] SounderPy: dimuat dari 'indeks_atmosfer_dataset.csv' -> 2774 baris, 29 kolom


In [5]:
# Audit 1-4: jumlah baris, jumlah kolom, daftar kolom, tipe data per kolom
structure_summary = {}

def audit_struktur(df, source_name):
    if df is None:
        return None
    info = {
        "total_rows": df.shape[0],
        "total_columns": df.shape[1],
        "columns": list(df.columns),
        "dtypes": df.dtypes.astype(str).to_dict(),
    }
    print(f"--- Struktur {source_name} ---")
    print(f"Jumlah baris : {info['total_rows']}")
    print(f"Jumlah kolom : {info['total_columns']}")
    print(f"Daftar kolom : {info['columns']}")
    print("Tipe data per kolom:")
    for col, dt in info["dtypes"].items():
        print(f"  - {col}: {dt}")
    print()
    structure_summary[source_name] = info
    return info


struct_ogimet = audit_struktur(df_ogimet, "Ogimet")
struct_sounderpy = audit_struktur(df_sounderpy, "SounderPy")

--- Struktur Ogimet ---
Jumlah baris : 3287
Jumlah kolom : 7
Daftar kolom : ['Date', 'Daily Rainfall (mm)', 'Average Air Temperature (°C)', 'Relative Humidity (%)', 'Pressure', 'Wind Speed', 'Wind Direction']
Tipe data per kolom:
  - Date: object
  - Daily Rainfall (mm): object
  - Average Air Temperature (°C): object
  - Relative Humidity (%): object
  - Pressure: object
  - Wind Speed: object
  - Wind Direction: object

--- Struktur SounderPy ---
Jumlah baris : 2774
Jumlah kolom : 29
Daftar kolom : ['observation_datetime', 'nominal_date', 'observation_hour', 'status', 'SBCAPE', 'SBCIN', 'MLCAPE', 'MLCIN', 'MUCAPE', 'MUCIN', 'DCAPE', 'MU_ECAPE', 'ML_ECAPE', 'SB_ECAPE', 'SFC_PRESSURE_hPa', 'SRH_0_1km', 'SRH_0_3km', 'SHEAR_0_6km_kt', 'EHI_0_3km', 'SCP', 'STP', 'LI_SB_500', 'TT', 'KINDEX', 'SWEAT', 'n_indices_computed', 'n_sharppy_direct_computed', 'reason', 'sharppy_direct_reason']
Tipe data per kolom:
  - observation_datetime: object
  - nominal_date: object
  - observation_hour: int64

In [6]:
# Audit 5-6: rentang tanggal minimum & maksimum (per sumber)
date_summary = {}

def audit_tanggal(df, source_name, date_col, start_year, end_year):
    if df is None:
        return None
    if date_col not in df.columns:
        print(f"[PERINGATAN] Kolom tanggal '{date_col}' tidak ditemukan di {source_name}.")
        date_summary[source_name] = None
        return None

    # Parsing dilakukan pada Series SEMENTARA -> TIDAK menimpa df asli
    parsed = pd.to_datetime(df[date_col], format="%d/%m/%Y", errors="coerce")
    n_valid = parsed.notna().sum()
    n_invalid = parsed.isna().sum()
    start_date = parsed.min()
    end_date = parsed.max()

    tahun_tersedia = sorted(parsed.dt.year.dropna().unique().tolist())
    tahun_diharapkan = list(range(start_year, end_year + 1))
    tahun_hilang = [t for t in tahun_diharapkan if t not in tahun_tersedia]

    print(f"--- Audit Tanggal {source_name} ---")
    print(f"Tanggal minimum   : {start_date}")
    print(f"Tanggal maksimum  : {end_date}")
    print(f"Tanggal valid     : {n_valid}")
    print(f"Tanggal invalid   : {n_invalid}")
    print(f"Tahun tersedia    : {tahun_tersedia}")
    print(f"Tahun HILANG (dari {start_year}-{end_year}): {tahun_hilang if tahun_hilang else 'Tidak ada'}")
    print()

    result = {
        "parsed_dates": parsed,   # dipakai cell berikut, TIDAK ditulis balik ke df
        "start_date": start_date,
        "end_date": end_date,
        "n_valid": int(n_valid),
        "n_invalid": int(n_invalid),
        "tahun_hilang": tahun_hilang,
    }
    date_summary[source_name] = result
    return result


date_result_ogimet = audit_tanggal(df_ogimet, "Ogimet", CONFIG["ogimet"]["date_col"],
                                    CONFIG["period"]["start_year"], CONFIG["period"]["end_year"])
date_result_sounderpy = audit_tanggal(df_sounderpy, "SounderPy", CONFIG["sounderpy"]["date_col"],
                                       CONFIG["period"]["start_year"], CONFIG["period"]["end_year"])

--- Audit Tanggal Ogimet ---
Tanggal minimum   : 2017-01-01 00:00:00
Tanggal maksimum  : 2024-12-31 00:00:00
Tanggal valid     : 2922
Tanggal invalid   : 365
Tahun tersedia    : [2017.0, 2018.0, 2019.0, 2020.0, 2021.0, 2022.0, 2023.0, 2024.0]
Tahun HILANG (dari 2017-2024): Tidak ada

--- Audit Tanggal SounderPy ---
Tanggal minimum   : 2017-01-01 00:00:00
Tanggal maksimum  : 2024-12-17 00:00:00
Tanggal valid     : 2774
Tanggal invalid   : 0
Tahun tersedia    : [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Tahun HILANG (dari 2017-2024): Tidak ada



In [7]:
# Audit 7: duplicate date (+ duplikat baris penuh sebagai info tambahan)
duplicate_summary = {}

def audit_duplikat(df, source_name, date_result):
    if df is None or date_result is None:
        return None
    parsed = date_result["parsed_dates"]
    n_dup_date = int(parsed.duplicated(keep=False).sum())
    n_dup_row_full = int(df.duplicated(keep=False).sum())

    print(f"--- Audit Duplikasi {source_name} ---")
    print(f"Baris dengan tanggal duplikat              : {n_dup_date}")
    print(f"Baris duplikat penuh (identik semua kolom) : {n_dup_row_full}")
    print()

    result = {"duplicate_dates": n_dup_date, "duplicate_full_rows": n_dup_row_full}
    duplicate_summary[source_name] = result
    return result


dup_ogimet = audit_duplikat(df_ogimet, "Ogimet", date_result_ogimet)
dup_sounderpy = audit_duplikat(df_sounderpy, "SounderPy", date_result_sounderpy)

--- Audit Duplikasi Ogimet ---
Baris dengan tanggal duplikat              : 365
Baris duplikat penuh (identik semua kolom) : 365

--- Audit Duplikasi SounderPy ---
Baris dengan tanggal duplikat              : 42
Baris duplikat penuh (identik semua kolom) : 0



In [8]:
# Audit 8-10: missing value per kolom, persentase missing per kolom, kolom kosong total
# Placeholder non-standar (mis. '----', '-----') ikut dihitung sebagai missing,
# tanpa mengubah nilai apa pun pada df.
missing_summary = {}

def audit_missing(df, source_name, missing_placeholders, threshold):
    if df is None:
        return None
    n_rows = len(df)

    na_count = df.isna().sum()

    placeholder_count = pd.Series(0, index=df.columns)
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            mask_exact = df[col].isin(missing_placeholders)
            # tangkap variasi placeholder berbasis dash (mis. '----', '-----', dst.)
            mask_dash_pattern = df[col].astype(str).str.fullmatch(r"-{2,}", na=False)
            mask = mask_exact | mask_dash_pattern
            placeholder_count[col] = mask.sum()

    total_missing_per_col = na_count + placeholder_count
    pct_missing_per_col = (total_missing_per_col / n_rows * 100).round(2)

    missing_df = pd.DataFrame({
        "kolom": df.columns,
        "n_missing_standar": na_count.values,
        "n_missing_placeholder": placeholder_count.values,
        "n_missing_total": total_missing_per_col.values,
        "pct_missing": pct_missing_per_col.values,
    }).sort_values("pct_missing", ascending=False)

    kolom_kosong_total = missing_df.loc[missing_df["pct_missing"] == 100.0, "kolom"].tolist()
    kolom_signifikan = missing_df.loc[
        (missing_df["pct_missing"] / 100 >= threshold) & (missing_df["pct_missing"] < 100.0), "kolom"
    ].tolist()

    total_missing_all = int(total_missing_per_col.sum())
    total_cells = n_rows * df.shape[1]
    pct_missing_all = round(total_missing_all / total_cells * 100, 2) if total_cells else 0.0

    print(f"--- Audit Missing Value {source_name} ---")
    print(missing_df.to_string(index=False))
    print(f"\nTotal missing (seluruh kolom): {total_missing_all} ({pct_missing_all}% dari seluruh sel)")
    print(f"Kolom KOSONG TOTAL (100% missing)          : {kolom_kosong_total if kolom_kosong_total else 'Tidak ada'}")
    print(f"Kolom missing SIGNIFIKAN (>= {int(threshold*100)}%) : {kolom_signifikan if kolom_signifikan else 'Tidak ada'}")
    print()

    result = {
        "detail": missing_df,
        "total_missing": total_missing_all,
        "pct_missing_all": pct_missing_all,
        "kolom_kosong_total": kolom_kosong_total,
        "kolom_signifikan": kolom_signifikan,
    }
    missing_summary[source_name] = result
    return result


missing_ogimet = audit_missing(df_ogimet, "Ogimet", CONFIG["ogimet"]["missing_placeholders"],
                                CONFIG["missing_significant_threshold"])
missing_sounderpy = audit_missing(df_sounderpy, "SounderPy", CONFIG["sounderpy"]["missing_placeholders"],
                                   CONFIG["missing_significant_threshold"])

--- Audit Missing Value Ogimet ---
                       kolom  n_missing_standar  n_missing_placeholder  n_missing_total  pct_missing
         Daily Rainfall (mm)                365                     37              402        12.23
Average Air Temperature (°C)                365                     31              396        12.05
       Relative Humidity (%)                365                     17              382        11.62
                  Wind Speed                365                     17              382        11.62
                    Pressure                365                     17              382        11.62
              Wind Direction                365                     17              382        11.62
                        Date                365                      0              365        11.10

Total missing (seluruh kolom): 2691 (11.7% dari seluruh sel)
Kolom KOSONG TOTAL (100% missing)          : Tidak ada
Kolom missing SIGNIFIKAN (>= 30%) : Tida

In [9]:
# Audit 11-12: kolom numerik yang masih terbaca sebagai teks/object,
# dan nilai non-numerik pada kolom yang seharusnya numerik.
# Menggunakan is_numeric_dtype (bukan '== object') agar konsisten di berbagai versi pandas.
numeric_summary = {}

def audit_numerik(df, source_name, column_map, missing_placeholders):
    if df is None:
        return None
    print(f"--- Validasi Kolom Numerik {source_name} ---")
    kolom_object_numeric = []
    non_numeric_detail = {}

    for var_name, col in column_map.items():
        if col not in df.columns:
            print(f"[DILEWATI] Kolom '{col}' (untuk variabel '{var_name}') tidak ditemukan.")
            continue

        dtype_awal = df[col].dtype
        if not pd.api.types.is_numeric_dtype(df[col]):
            kolom_object_numeric.append(col)

        # Percobaan konversi dilakukan pada SERIES SEMENTARA (bukan df asli)
        original_non_null = df[col].notna() & (~df[col].isin(missing_placeholders))
        converted = pd.to_numeric(df[col], errors="coerce")
        n_non_numeric = int((original_non_null & converted.isna()).sum())
        contoh = df.loc[original_non_null & converted.isna(), col].unique()[:5].tolist()

        non_numeric_detail[col] = n_non_numeric

        status = f"{dtype_awal} (perlu konversi)" if not pd.api.types.is_numeric_dtype(df[col]) else str(dtype_awal)
        print(f"'{var_name}' -> kolom '{col}' | tipe awal: {status} | "
              f"nilai non-numerik: {n_non_numeric}"
              + (f" | contoh: {contoh}" if n_non_numeric else ""))
    print()

    result = {
        "kolom_object_numeric": kolom_object_numeric,
        "non_numeric_detail": non_numeric_detail,
    }
    numeric_summary[source_name] = result
    return result


numeric_ogimet = audit_numerik(df_ogimet, "Ogimet", CONFIG["ogimet"]["column_map"],
                                CONFIG["ogimet"]["missing_placeholders"])
numeric_sounderpy = audit_numerik(df_sounderpy, "SounderPy", CONFIG["sounderpy"]["column_map"],
                                   CONFIG["sounderpy"]["missing_placeholders"])

--- Validasi Kolom Numerik Ogimet ---
'RR' -> kolom 'Daily Rainfall (mm)' | tipe awal: object (perlu konversi) | nilai non-numerik: 184 | contoh: ['Tr']
'Tavg' -> kolom 'Average Air Temperature (°C)' | tipe awal: object (perlu konversi) | nilai non-numerik: 31 | contoh: ['-----']
'RH' -> kolom 'Relative Humidity (%)' | tipe awal: object (perlu konversi) | nilai non-numerik: 17 | contoh: ['-----']

--- Validasi Kolom Numerik SounderPy ---
'CIN' -> kolom 'SBCIN' | tipe awal: float64 | nilai non-numerik: 0
'K-Index' -> kolom 'KINDEX' | tipe awal: float64 | nilai non-numerik: 0
'LI' -> kolom 'LI_SB_500' | tipe awal: float64 | nilai non-numerik: 0
'TT' -> kolom 'TT' | tipe awal: float64 | nilai non-numerik: 0
'SWEAT' -> kolom 'SWEAT' | tipe awal: object (perlu konversi) | nilai non-numerik: 1 | contoh: ['2.625.196']



In [10]:
# Audit 14-16 (khusus SounderPy): distribusi observation_hour, distribusi status,
# persentase SUCCESS vs non-SUCCESS.
sounding_summary = {}

def audit_sounding(df, source_name, hour_col, status_col):
    if df is None:
        return None
    print(f"--- Audit Sounding {source_name} ---")
    result = {}

    if hour_col in df.columns:
        dist_hour = df[hour_col].value_counts(dropna=False)
        print("Distribusi observation_hour:")
        print(dist_hour.to_string())
        result["distribusi_hour"] = dist_hour.to_dict()
    else:
        print(f"[PERINGATAN] Kolom '{hour_col}' tidak ditemukan.")
        result["distribusi_hour"] = None

    if status_col in df.columns:
        dist_status = df[status_col].value_counts(dropna=False)
        print("\nDistribusi status:")
        print(dist_status.to_string())

        n_total = len(df)
        n_success = int((df[status_col] == "SUCCESS").sum())
        n_non_success = n_total - n_success
        pct_success = round(n_success / n_total * 100, 2) if n_total else 0.0
        pct_non_success = round(n_non_success / n_total * 100, 2) if n_total else 0.0

        print(f"\nSUCCESS     : {n_success} ({pct_success}%)")
        print(f"NON-SUCCESS : {n_non_success} ({pct_non_success}%)")

        result["distribusi_status"] = dist_status.to_dict()
        result["pct_success"] = pct_success
        result["pct_non_success"] = pct_non_success
    else:
        print(f"[PERINGATAN] Kolom '{status_col}' tidak ditemukan.")
        result["distribusi_status"] = None

    print()
    sounding_summary[source_name] = result
    return result


sounding_result = audit_sounding(df_sounderpy, "SounderPy",
                                  CONFIG["sounderpy"]["hour_col"], CONFIG["sounderpy"]["status_col"])

--- Audit Sounding SounderPy ---
Distribusi observation_hour:
observation_hour
12    2314
0      460

Distribusi status:
status
SUCCESS    2774

SUCCESS     : 2774 (100.0%)
NON-SUCCESS : 0 (0.0%)



In [11]:
# Audit 13: ringkasan temuan audit -> disusun jadi satu tabel gabungan (Ogimet + SounderPy)
def build_notes(source_name):
    parts = []

    if missing_summary.get(source_name):
        kolom_kosong = missing_summary[source_name]["kolom_kosong_total"]
        kolom_signifikan = missing_summary[source_name]["kolom_signifikan"]
        parts.append(f"kolom_kosong_total={kolom_kosong if kolom_kosong else 'tidak ada'}")
        parts.append(f"kolom_missing_signifikan={kolom_signifikan if kolom_signifikan else 'tidak ada'}")

    if numeric_summary.get(source_name):
        kolom_obj = numeric_summary[source_name]["kolom_object_numeric"]
        non_numeric = numeric_summary[source_name]["non_numeric_detail"]
        parts.append(f"kolom_numerik_masih_object={kolom_obj if kolom_obj else 'tidak ada'}")
        temuan_non_numeric = {k: v for k, v in non_numeric.items() if v > 0}
        parts.append(f"nilai_non_numerik={temuan_non_numeric if temuan_non_numeric else 'tidak ada'}")

    if duplicate_summary.get(source_name):
        parts.append(f"duplicate_full_rows={duplicate_summary[source_name]['duplicate_full_rows']}")

    if date_summary.get(source_name):
        th = date_summary[source_name]["tahun_hilang"]
        parts.append(f"tahun_hilang={th if th else 'tidak ada'}")

    if source_name == "SounderPy":
        parts.append("catatan_format_angka=beberapa kolom numerik (mis. SBCAPE) mengandung "
                     "pola titik ganda seperti '2.226.401' yang mengindikasikan pemisah "
                     "ribuan/desimal tidak konsisten; perlu aturan parsing eksplisit di Tahap 2")
        if sounding_summary.get(source_name):
            parts.append(f"pct_success={sounding_summary[source_name].get('pct_success')}")
            parts.append(f"pct_non_success={sounding_summary[source_name].get('pct_non_success')}")

    return " | ".join(parts)


def build_report_row(source_name, df, struct, date_res, dup_res, miss_res):
    if df is None:
        return {
            "source": source_name,
            "total_rows": 0,
            "total_columns": 0,
            "start_date": None,
            "end_date": None,
            "duplicate_dates": None,
            "missing_values": None,
            "missing_percentage": None,
            "notes": "GAGAL DIMUAT - file tidak ditemukan/error saat load",
        }

    return {
        "source": source_name,
        "total_rows": struct["total_rows"],
        "total_columns": struct["total_columns"],
        "start_date": date_res["start_date"].date() if date_res and pd.notna(date_res["start_date"]) else None,
        "end_date": date_res["end_date"].date() if date_res and pd.notna(date_res["end_date"]) else None,
        "duplicate_dates": dup_res["duplicate_dates"] if dup_res else None,
        "missing_values": miss_res["total_missing"] if miss_res else None,
        "missing_percentage": miss_res["pct_missing_all"] if miss_res else None,
        "notes": build_notes(source_name),
    }


report_rows = [
    build_report_row("Ogimet", df_ogimet, struct_ogimet, date_result_ogimet, dup_ogimet, missing_ogimet),
    build_report_row("SounderPy", df_sounderpy, struct_sounderpy, date_result_sounderpy, dup_sounderpy, missing_sounderpy),
]

audit_report_df = pd.DataFrame(report_rows, columns=[
    "source", "total_rows", "total_columns", "start_date", "end_date",
    "duplicate_dates", "missing_values", "missing_percentage", "notes"
])

print("--- Ringkasan Audit Report ---")
display(audit_report_df)

--- Ringkasan Audit Report ---


,source,total_rows,total_columns,start_date,end_date,duplicate_dates,missing_values,missing_percentage,notes
0,Ogimet,3287,7,2017-01-01,2024-12-31,365,2691,11.70,kolom_kosong_total=tidak ada | kolom_missing_s...
1,SounderPy,2774,29,2017-01-01,2024-12-17,42,9713,12.07,"kolom_kosong_total=['sharppy_direct_reason', '..."


In [12]:
audit_report_df.to_csv(CONFIG["output_path"], index=False)
print(f"[TERSIMPAN] {CONFIG['output_path']} -> {os.path.abspath(CONFIG['output_path'])}")

# Verifikasi read-only: pastikan file input asal tidak tersentuh oleh notebook ini
print("\nVerifikasi berkas asal tidak berubah:")
for cfg, name in [(CONFIG["ogimet"], "Ogimet"), (CONFIG["sounderpy"], "SounderPy")]:
    if os.path.exists(cfg["path"]):
        size = os.path.getsize(cfg["path"])
        print(f"  {name}: {cfg['path']} ({size} bytes) - hanya dibaca, tidak ditimpa")

[TERSIMPAN] 01_audit_report.csv -> /content/01_audit_report.csv

Verifikasi berkas asal tidak berubah:
  Ogimet: ogimet_dataset.csv (121457 bytes) - hanya dibaca, tidak ditimpa
  SounderPy: indeks_atmosfer_dataset.csv (530001 bytes) - hanya dibaca, tidak ditimpa


In [13]:
print("=" * 70)
print("VALIDATION CHECKLIST — TAHAP 1: AUDIT DATASET MENTAH")
print("=" * 70)

# --- Validasi (playbook) ---
cek_tahun_ogimet = not date_result_ogimet["tahun_hilang"] if date_result_ogimet else False
cek_tahun_sounderpy = not date_result_sounderpy["tahun_hilang"] if date_result_sounderpy else False
cek_rentang_tahun_ok = cek_tahun_ogimet and cek_tahun_sounderpy

kolom_inti_ogimet = list(CONFIG["ogimet"]["column_map"].values())
kolom_inti_sounderpy = list(CONFIG["sounderpy"]["column_map"].values())

def cek_kolom_inti_ada(df, kolom_list):
    if df is None:
        return False
    return all(col in df.columns for col in kolom_list)

cek_kolom_inti_ok = (cek_kolom_inti_ada(df_ogimet, kolom_inti_ogimet) and
                      cek_kolom_inti_ada(df_sounderpy, kolom_inti_sounderpy))

print("\n[VALIDASI]")
print(f"[{'v' if cek_rentang_tahun_ok else 'x'}] Rentang tahun mencakup 2017-2024 secara utuh untuk kedua sumber")
if not cek_tahun_ogimet:
    print(f"      -> Ogimet: tahun hilang {date_result_ogimet['tahun_hilang']}")
if not cek_tahun_sounderpy:
    print(f"      -> SounderPy: tahun hilang {date_result_sounderpy['tahun_hilang']}")
print(f"[{'v' if cek_kolom_inti_ok else 'x'}] Tidak ada kolom variabel inti yang hilang total dari file")

# --- Checkpoint Sebelum Lanjut (playbook) ---
cek_output_dihasilkan = os.path.exists(CONFIG["output_path"])
cek_load_ok = load_status.get("Ogimet", False) and load_status.get("SounderPy", False)

print("\n[CHECKPOINT SEBELUM LANJUT]")
print(f"[{'v' if cek_output_dihasilkan else 'x'}] 01_audit_report.csv dihasilkan")
print(f"[v] Rentang tahun & struktur kolom terdokumentasi (tersimpan pada audit_report_df & console di atas)")
print(f"[{'v' if cek_load_ok else 'x'}] Tidak ada sumber data yang gagal dimuat")

print("\n" + "=" * 70)
if cek_output_dihasilkan and cek_load_ok and cek_rentang_tahun_ok and cek_kolom_inti_ok:
    print("STATUS: Tahap 1 SELESAI - seluruh checkpoint dan validasi terpenuhi.")
else:
    print("STATUS: Tahap 1 BELUM SEPENUHNYA memenuhi checklist - tinjau item bertanda [x].")
print("=" * 70)

VALIDATION CHECKLIST — TAHAP 1: AUDIT DATASET MENTAH

[VALIDASI]
[v] Rentang tahun mencakup 2017-2024 secara utuh untuk kedua sumber
[v] Tidak ada kolom variabel inti yang hilang total dari file

[CHECKPOINT SEBELUM LANJUT]
[v] 01_audit_report.csv dihasilkan
[v] Rentang tahun & struktur kolom terdokumentasi (tersimpan pada audit_report_df & console di atas)
[v] Tidak ada sumber data yang gagal dimuat

STATUS: Tahap 1 SELESAI - seluruh checkpoint dan validasi terpenuhi.
